# Taller 1 - Sistema masa-resorte-amortiguador (2 GDL)

Este notebook desarrolla el procedimiento del Taller 1 para un sistema de dos grados de libertad con excitacion de base.

Se usa la libreria `python-control` para modelar y simular la respuesta dinamica.

In [ ]:
!pip install control

In [ ]:
import control as sc
import numpy as np
import matplotlib.pyplot as plt

## 1. Planteamiento del modelo

Ecuaciones del sistema:

$$
m_1 \ddot{x}_1 + c(\dot{x}_1-\dot{x}_2) + k_2(x_1-x_2) + k_1(x_1-y)=0
$$

$$
m_2 \ddot{x}_2 + c(\dot{x}_2-\dot{x}_1) + k_2(x_2-x_1)=0
$$

con entrada de base:

$$y(t)=A\sin(\omega t)$$

In [ ]:
# Parametros fisicos (puedes ajustarlos segun tu taller)
m1 = 250.0
m2 = 1000.0
k1 = 16000.0
k2 = 12000.0
c = 1000.0

# Entrada de base y(t)=A*sin(omega*t)
Ain = 0.05
omega = 4.0

# Tiempo de simulacion
t = np.linspace(0, 20, 4000)

# Condiciones iniciales
X0 = np.array([0.0, 0.0, 0.0, 0.0])

print('Parametros cargados correctamente.')

## 2. Modelo en espacio de estados

Definimos los estados:

$$x=[x_1, x_2, \dot{x}_1, \dot{x}_2]^T$$

y tomamos como entrada el desplazamiento de base $u=y(t)$.

Ademas, para representar la excitacion de base en la dinamica se usa:

$$\dot{v}_1 = \ddot{x}_1 = \frac{-c(v_1-v_2)-k_2(x_1-x_2)-k_1(x_1-u)}{m_1}$$

$$\dot{v}_2 = \ddot{x}_2 = \frac{-c(v_2-v_1)-k_2(x_2-x_1)}{m_2}$$

In [ ]:
# Matrices del sistema lineal xdot = A x + B u
A = np.array([
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [-(k1+k2)/m1, k2/m1, -c/m1, c/m1],
    [k2/m2, -k2/m2, c/m2, -c/m2]
], dtype=float)

B = np.array([[0], [0], [k1/m1], [0]], dtype=float)

# Salidas: x1 y x2
C = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], dtype=float)

D = np.array([[0], [0]], dtype=float)

sys_ss = sc.ss(A, B, C, D)
print('Modelo en espacio de estados:')
print(sys_ss)

## 3. Simulacion con entrada senoidal de base

Se simula con:

$$u(t)=A_{in}\sin(\omega t)$$

In [ ]:
u = Ain * np.sin(omega * t)

# Respuesta forzada del sistema
tout, yout, xout = sc.forced_response(sys_ss, T=t, U=u, X0=X0, return_x=True)

x1 = yout[0, :]
x2 = yout[1, :]

# Graficas principales
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

ax[0].plot(tout, u, 'k', linewidth=1.4)
ax[0].set_title('Entrada de base y(t)')
ax[0].set_ylabel('m')
ax[0].grid(True)

ax[1].plot(tout, x1, 'b', linewidth=1.5)
ax[1].set_title('Desplazamiento x1(t) - masa m1')
ax[1].set_ylabel('m')
ax[1].grid(True)

ax[2].plot(tout, x2, 'r', linewidth=1.5)
ax[2].set_title('Desplazamiento x2(t) - masa m2')
ax[2].set_ylabel('m')
ax[2].set_xlabel('Tiempo (s)')
ax[2].grid(True)

plt.tight_layout()
plt.show()

## 4. Analisis rapido de desempeno

Se calcula amplitud pico en estado estacionario aproximado para comparar vibracion en `m1` y `m2`.

In [ ]:
# Ventana final para aproximar estado estacionario
mask = tout > (0.7 * tout[-1])

amp_x1 = 0.5 * (np.max(x1[mask]) - np.min(x1[mask]))
amp_x2 = 0.5 * (np.max(x2[mask]) - np.min(x2[mask]))
amp_u = 0.5 * (np.max(u[mask]) - np.min(u[mask]))

print('Amplitud entrada y(t):', round(amp_u, 6), 'm')
print('Amplitud x1(t):       ', round(amp_x1, 6), 'm')
print('Amplitud x2(t):       ', round(amp_x2, 6), 'm')

if amp_x2 < amp_x1:
    print('Interpretacion: el chasis (m2) vibra menos que la rueda (m1).')
else:
    print('Interpretacion: revisar parametros, m2 no esta atenuando respecto a m1.')

## 5. Exploracion parametrica (opcional)

Esta celda permite variar `c` para observar su efecto en la vibracion del chasis.

In [ ]:
c_values = [500, 1000, 2000, 4000]
amp_x2_list = []

for c_test in c_values:
    A_test = np.array([
        [0, 0, 1, 0],
        [0, 0, 0, 1],
        [-(k1+k2)/m1, k2/m1, -c_test/m1, c_test/m1],
        [k2/m2, -k2/m2, c_test/m2, -c_test/m2]
    ], dtype=float)

    sys_test = sc.ss(A_test, B, C, D)
    t_test, y_test = sc.forced_response(sys_test, T=t, U=u)
    x2_test = y_test[1, :]

    mask_test = t_test > (0.7 * t_test[-1])
    amp_test = 0.5 * (np.max(x2_test[mask_test]) - np.min(x2_test[mask_test]))
    amp_x2_list.append(amp_test)

plt.figure(figsize=(8, 4.5))
plt.plot(c_values, amp_x2_list, 'o-', linewidth=1.8)
plt.title('Efecto del amortiguamiento c en amplitud de x2')
plt.xlabel('c [N*s/m]')
plt.ylabel('Amplitud estacionaria de x2 [m]')
plt.grid(True)
plt.show()

for ci, ai in zip(c_values, amp_x2_list):
    print(f'c={ci:>4} -> amp_x2={ai:.6f} m')

## 6. Conclusiones

- Se construyo el modelo dinamico en forma de espacio de estados.
- Se simulo la respuesta ante excitacion senoidal de base con `python-control`.
- Se evaluo el comportamiento de `x1` y `x2` y el efecto del amortiguamiento.

Ajusta los parametros a los datos oficiales de tu guia para generar resultados finales del informe.